In [ ]:
# PARAMETERS CELL
SETTINGS = [
    "qwen/gsm8k",
    "qwen/hotpotqa",
]

DRY_RUN_COUNT = 20
FULL_RUN_COUNT = 500
OUTPUT_DIR = "/kaggle/working"


In [ ]:
import subprocess, sys, os

print("=" * 60)
print("1. INSTALLING UV & CREATING ISOLATED VIRTUAL ENVIRONMENT")
print("=" * 60)

# Set HF_HOME to /tmp/hf so model weights do not clutter /kaggle/working
os.environ["HF_HOME"] = "/tmp/hf"
os.makedirs("/tmp/hf", exist_ok=True)

# Install uv into system python (fast, standalone, doesn't touch system libraries)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])

# Create clean virtual environment with Python 3.11 in /tmp/venv
subprocess.check_call(["uv", "venv", "/tmp/venv", "--python", "3.11"])

# Install vllm and dependencies inside /tmp/venv
print("\nInstalling vllm and dependencies inside /tmp/venv...")
res = subprocess.run([
    "uv", "pip", "install", "--python", "/tmp/venv/bin/python",
    "vllm==0.6.6.post1", "setuptools", "transformers==4.46.1", "pyarrow<19.0.0,>=15.0.0", "datasets>=3.0.0", "pandas", "scipy", "scikit-learn", "pyyaml"
])
if res.returncode != 0:
    print("Fallback: Installing vllm==0.7.3 into /tmp/venv...")
    subprocess.check_call([
        "uv", "pip", "install", "--python", "/tmp/venv/bin/python",
        "vllm==0.7.3", "setuptools", "transformers==4.46.1", "pyarrow<19.0.0,>=15.0.0", "datasets>=3.0.0", "pandas", "scipy", "scikit-learn", "pyyaml"
    ])

print("\nIsolated virtual environment ready at /tmp/venv.")


In [ ]:
import subprocess, sys

print("=" * 60)
print("2. VIRTUAL ENVIRONMENT & HARDWARE VERIFICATION")
print("=" * 60)
subprocess.run([
    "/tmp/venv/bin/python", "-c",
    "import numpy, torch, vllm, transformers; print(f'PyTorch: {torch.__version__}, NumPy: {numpy.__version__}, vLLM: {vllm.__version__}, Transformers: {transformers.__version__}')"
], check=True)

print("\nNVIDIA-SMI OUTPUT:")
subprocess.run(["nvidia-smi"], check=True)
print("=" * 60)
sys.stdout.flush()


In [ ]:
# 3. PREPARE CGSC PACKAGE FILES & DATASETS
import os, sys, shutil, gzip, base64

os.makedirs("cgsc/data", exist_ok=True)
os.makedirs("cgsc/prompts", exist_ok=True)
os.makedirs("results/raw", exist_ok=True)

with open("cgsc/__init__.py", "w", encoding="utf-8") as f:
    f.write('"""CGSC Package"""\n')

with open("cgsc/scoring.py", "w", encoding="utf-8") as f:
    f.write('"""Scoring and answer extraction for GSM8K and HotpotQA."""\n\nimport re\nimport string\nfrom collections import Counter\nfrom typing import Tuple, Optional\n\n\ndef extract_final_answer(text: str) -> str:\n    """Extract text after the LAST \'Final answer:\' (case-insensitive).\n    If absent, return the last non-empty line."""\n    if not text:\n        return ""\n    matches = list(re.finditer(r"final\\s+answer\\s*:\\s*", text, flags=re.IGNORECASE))\n    if matches:\n        last_match = matches[-1]\n        extracted = text[last_match.end():].strip()\n        # Take the first line or strip subsequent blank lines\n        lines = [line.strip() for line in extracted.splitlines() if line.strip()]\n        return lines[0] if lines else extracted\n    \n    # If absent, use the last non-empty line\n    lines = [line.strip() for line in text.splitlines() if line.strip()]\n    return lines[-1] if lines else ""\n\n\ndef normalise_gsm8k(text: str) -> Optional[float]:\n    """Take the last number in the extracted span; strip \'$\', \',\', \'%\', spaces.\n    Return as float if valid, else None."""\n    if not text:\n        return None\n    cleaned = text.replace(",", "").replace("$", "").replace("%", " ")\n    # Find all numeric patterns (integers, floats, negative numbers)\n    numbers = re.findall(r"[-+]?(?:\\d*\\.\\d+|\\d+)", cleaned)\n    if not numbers:\n        return None\n    try:\n        return float(numbers[-1])\n    except ValueError:\n        return None\n\n\ndef score_gsm8k(extracted_answer: str, gold_answer: str, tol: float = 1e-6) -> Tuple[bool, bool, float]:\n    """Score GSM8K: returns (em, lenient, f1).\n    Float-compare with tolerance 1e-6 (so 12, 12.0, $12 all equal 12).\n    For GSM8K, lenient and f1 match em."""\n    pred_num = normalise_gsm8k(extracted_answer)\n    gold_num = normalise_gsm8k(gold_answer)\n    if pred_num is None or gold_num is None:\n        return False, False, 0.0\n    match = abs(pred_num - gold_num) <= tol\n    return match, match, 1.0 if match else 0.0\n\n\ndef normalise_hotpotqa(s: str) -> str:\n    """Official SQuAD / HotpotQA answer normalisation:\n    Lower-case, remove punctuation, remove articles (a, an, the), and collapse extra whitespace."""\n    def remove_articles(text: str) -> str:\n        return re.sub(r"\\b(a|an|the)\\b", " ", text)\n\n    def white_space_fix(text: str) -> str:\n        return " ".join(text.split())\n\n    def remove_punc(text: str) -> str:\n        exclude = set(string.punctuation)\n        return "".join(ch for ch in text if ch not in exclude)\n\n    def lower(text: str) -> str:\n        return text.lower()\n\n    return white_space_fix(remove_articles(remove_punc(lower(s))))\n\n\ndef exact_match_score(prediction: str, ground_truth: str) -> bool:\n    """EM on normalised strings."""\n    return normalise_hotpotqa(prediction) == normalise_hotpotqa(ground_truth)\n\n\ndef lenient_match_score(prediction: str, ground_truth: str) -> bool:\n    """Lenient match: normalised gold answer is contained in normalised extracted answer."""\n    norm_gold = normalise_hotpotqa(ground_truth)\n    norm_pred = normalise_hotpotqa(prediction)\n    if not norm_gold:\n        return False\n    return norm_gold in norm_pred\n\n\ndef f1_score(prediction: str, ground_truth: str) -> float:\n    """Token-level F1 on normalised strings."""\n    norm_pred = normalise_hotpotqa(prediction)\n    norm_gold = normalise_hotpotqa(ground_truth)\n    pred_tokens = norm_pred.split()\n    gold_tokens = norm_gold.split()\n    common = Counter(pred_tokens) & Counter(gold_tokens)\n    num_same = sum(common.values())\n    if len(pred_tokens) == 0 or len(gold_tokens) == 0:\n        # If either is no-answer, then F1 is 1 if they agree, 0 otherwise\n        return int(pred_tokens == gold_tokens)\n    if num_same == 0:\n        return 0.0\n    precision = 1.0 * num_same / len(pred_tokens)\n    recall = 1.0 * num_same / len(gold_tokens)\n    return (2 * precision * recall) / (precision + recall)\n\n\ndef score_hotpotqa(extracted_answer: str, gold_answer: str) -> Tuple[bool, bool, float]:\n    """Score HotpotQA: returns (em, lenient, f1)."""\n    em = exact_match_score(extracted_answer, gold_answer)\n    lenient = lenient_match_score(extracted_answer, gold_answer)\n    f1 = f1_score(extracted_answer, gold_answer)\n    return em, lenient, f1\n\n\ndef majority_vote(answers: list) -> str:\n    """Find most frequent normalised extracted answer among A0 + samples.\n    Ties broken in favour of A0\'s answer (which is index 0)."""\n    if not answers:\n        return ""\n    norm_counts = Counter()\n    norm_to_orig = {}\n    for ans in reversed(answers):  # so earlier items overwrite in norm_to_orig\n        norm = normalise_hotpotqa(ans)\n        norm_to_orig[norm] = ans\n    for ans in answers:\n        norm = normalise_hotpotqa(ans)\n        norm_counts[norm] += 1\n    \n    # Tie breaking: if A0 is tied for max, pick A0\n    a0_norm = normalise_hotpotqa(answers[0])\n    max_count = max(norm_counts.values())\n    if norm_counts[a0_norm] == max_count:\n        return answers[0]\n    \n    # Otherwise return the most frequent\n    best_norm, _ = norm_counts.most_common(1)[0]\n    return norm_to_orig[best_norm]\n')

with open("cgsc/sanity.py", "w", encoding="utf-8") as f:
    f.write('"""Built-in sanity checks for confidence-gated self-correction dry run."""\n\nimport json\nfrom typing import List, Dict, Any, Tuple\nimport numpy as np\n\n\ndef compute_auroc_binary(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    """Compute AUROC without external dependencies using trapezoidal rule on ranks."""\n    y_true = np.asarray(y_true, dtype=bool)\n    y_score = np.asarray(y_score, dtype=float)\n    n_pos = np.sum(y_true)\n    n_neg = len(y_true) - n_pos\n    if n_pos == 0 or n_neg == 0:\n        return 0.5\n    # Rank scores with average for ties\n    ranks = np.argsort(np.argsort(y_score)) + 1\n    # Mann-Whitney U test statistic for positive class\n    sum_ranks_pos = np.sum(ranks[y_true])\n    u_stat = sum_ranks_pos - (n_pos * (n_pos + 1)) / 2.0\n    return float(u_stat / (n_pos * n_neg))\n\n\ndef run_sanity_checks(records: List[Dict[str, Any]], model_name: str, dataset_name: str) -> Tuple[bool, str]:\n    """Run all 7 sanity checks specified in Step 5 on dry-run records.\n    Returns (all_passed, report_text)."""\n    lines = []\n    lines.append("=" * 80)\n    lines.append(f" SANITY CHECK REPORT: Model={model_name}, Dataset={dataset_name} (N={len(records)})")\n    lines.append("=" * 80)\n\n    overall_pass = True\n\n    # Check 1: LEAK check\n    # Assert gold answer does NOT appear in S_verif prompt unless in question/context or extracted A0\n    leak_fail = 0\n    leak_examples = []\n    for r in records:\n        gold = str(r.get("gold", "")).strip().lower()\n        verif_prompt = str(r.get("verif_prompt", "")).lower()\n        q_ctx = (str(r.get("question", "")) + " " + str(r.get("context", ""))).lower()\n        a0_ans = str(r.get("a0_extracted", "")).lower()\n\n        # Only check if gold is meaningful (at least 2 chars)\n        if len(gold) >= 2 and gold in verif_prompt:\n            if gold not in q_ctx and gold not in a0_ans:\n                leak_fail += 1\n\n    leak_status = "PASS" if leak_fail == 0 else "FAIL"\n    if leak_fail > 0:\n        overall_pass = False\n    lines.append(f"\\n[1] LEAK Check: {leak_status}")\n    lines.append(f"    Gold leaks detected in S_verif prompt: {leak_fail}/{len(records)}")\n    lines.append("    Printing 2 sample S_verif prompts:")\n    for i, r in enumerate(records[:2], 1):\n        vp = str(r.get("verif_prompt", ""))\n        # Truncate context if long\n        if len(vp) > 400:\n            vp_disp = vp[:300] + "\\n... [truncated] ...\\n" + vp[-100:]\n        else:\n            vp_disp = vp\n        lines.append(f"    --- Example {i} ---")\n        for pline in vp_disp.splitlines():\n            lines.append(f"    | {pline}")\n\n    # Check 2: S_verif computed from A0\'s extracted answer\n    verif_match_fail = 0\n    for r in records:\n        a0_ext = str(r.get("a0_extracted", "")).strip()\n        verif_prop = str(r.get("verif_proposed_answer", "")).strip()\n        if a0_ext != verif_prop:\n            verif_match_fail += 1\n    vmatch_status = "PASS" if verif_match_fail == 0 else "FAIL"\n    if verif_match_fail > 0:\n        overall_pass = False\n    lines.append(f"\\n[2] S_verif Alignment with A0: {vmatch_status}")\n    lines.append(f"    Mismatches between A0 extracted answer and verif proposed answer: {verif_match_fail}")\n\n    # Check 3: ID Alignment\n    id_list = [r["id"] for r in records if "id" in r]\n    unique_ids = set(id_list)\n    id_status = "PASS" if len(id_list) == len(records) and len(unique_ids) == len(records) else "FAIL"\n    if id_status == "FAIL":\n        overall_pass = False\n    lines.append(f"\\n[3] ID Alignment: {id_status}")\n    lines.append(f"    Total records: {len(records)}, Unique IDs: {len(unique_ids)}")\n\n    # Check 4: Random baseline AUROC vs Y0 within [0.35, 0.65]\n    y0_list = [int(r.get("y0", 0)) for r in records]\n    rng = np.random.default_rng(0)\n    rand_scores = rng.random(len(records))\n    rand_auroc = compute_auroc_binary(np.array(y0_list), rand_scores)\n    rand_pass = 0.35 <= rand_auroc <= 0.65\n    rand_status = "PASS" if rand_pass else "FAIL"\n    if not rand_pass:\n        overall_pass = False\n    lines.append(f"\\n[4] Random Baseline AUROC: {rand_status}")\n    lines.append(f"    Random AUROC vs Y0: {rand_auroc:.4f} (Expected in [0.35, 0.65])")\n\n    # Check 5: Print 3 examples: A0 answer, 10 sample answers, S_sc10\n    lines.append(f"\\n[5] Self-Consistency Agreement Examples (3 samples):")\n    for i, r in enumerate(records[:3], 1):\n        a0_ans = r.get("a0_extracted", "")\n        sc_ans = r.get("sc_extracted_answers", [])\n        sc10 = r.get("s_sc10", 0.0)\n        lines.append(f"    --- Example {i} (ID: {r.get(\'id\', \'\')}) ---")\n        lines.append(f"    A0 answer: {a0_ans}")\n        lines.append(f"    10 sample answers: {sc_ans}")\n        lines.append(f"    S_sc10 agreement: {sc10:.2f}")\n\n    # Check 6: Parse-failure counts\n    verb_fails = sum(1 for r in records if r.get("verb_parse_fail", False))\n    verif_fails = sum(1 for r in records if r.get("verif_parse_fail", False))\n    lines.append(f"\\n[6] Parse Failures:")\n    lines.append(f"    S_verb parse failures: {verb_fails}/{len(records)}")\n    lines.append(f"    S_verif parse failures: {verif_fails}/{len(records)}")\n\n    # Check 7: Timing and projected GPU-hours\n    time_per_q = [r.get("duration_seconds", 0.0) for r in records if "duration_seconds" in r]\n    avg_sec = np.mean(time_per_q) if time_per_q else 0.0\n    # Full run is 500 questions per setting\n    projected_gpu_hours = (avg_sec * 500.0) / 3600.0\n    lines.append(f"\\n[7] Timing & Projected Compute:")\n    lines.append(f"    Mean latency per question: {avg_sec:.2f} seconds")\n    lines.append(f"    Projected full-run time (500 questions): {projected_gpu_hours:.2f} GPU-hours")\n    time_pass = projected_gpu_hours <= 10.0\n    time_status = "PASS" if time_pass else "WARN (exceeds 10 hours)"\n    lines.append(f"    Status: {time_status}")\n\n    # Check 8: Sample Diversity Check (S_sc)\n    all_raw_identical_count = 0\n    extracted_distinct_counts = []\n    for r in records:\n        raw_texts = r.get("sc_raw_texts")\n        if not raw_texts:\n            raw_texts = r.get("sc_extracted_answers", [])\n        if len(set(raw_texts)) <= 1:\n            all_raw_identical_count += 1\n        \n        ext_answers = r.get("sc_extracted_answers", [])\n        extracted_distinct_counts.append(len(set(ext_answers)))\n\n    frac_identical = all_raw_identical_count / max(1, len(records))\n    sc_diversity_pass = frac_identical <= 0.50\n    if not sc_diversity_pass:\n        overall_pass = False\n\n    from collections import Counter\n    dist_map = Counter(extracted_distinct_counts)\n    dist_str = ", ".join(f"{k} distinct: {dist_map[k]}" for k in sorted(dist_map.keys()))\n\n    lines.append(f"\\n[8] Sample Diversity Check (S_sc): {\'PASS\' if sc_diversity_pass else \'FAIL\'}")\n    lines.append(f"    Questions with all 10 raw sample texts identical: {all_raw_identical_count}/{len(records)} ({frac_identical*100:.1f}%, threshold <= 50.0%)")\n    lines.append(f"    Distribution of distinct extracted answers per question: [{dist_str}]")\n\n    lines.append("\\n" + "=" * 80)\n    lines.append(f" OVERALL VERDICT: {\'ALL PASS\' if overall_pass else \'FAIL\'}")\n    lines.append("=" * 80)\n\n    report_text = "\\n".join(lines)\n    return overall_pass, report_text\n\n\nif __name__ == "__main__":\n    import sys\n    if len(sys.argv) > 1:\n        path = sys.argv[1]\n        with open(path, "r", encoding="utf-8") as f:\n            data = [json.loads(l) for l in f if l.strip()]\n        passed, report = run_sanity_checks(data, "model", "dataset")\n        print(report)\n        sys.exit(0 if passed else 1)\n')

with open("cgsc/data/splits.json", "w", encoding="utf-8") as f:
    f.write('{\n  "seed": 42,\n  "n_sample": 500,\n  "n_dev": 100,\n  "n_test": 400,\n  "gsm8k": {\n    "dev": [\n      "gsm8k-test-106",\n      "gsm8k-test-93",\n      "gsm8k-test-65",\n      "gsm8k-test-707",\n      "gsm8k-test-1039",\n      "gsm8k-test-1133",\n      "gsm8k-test-96",\n      "gsm8k-test-546",\n      "gsm8k-test-77",\n      "gsm8k-test-1266",\n      "gsm8k-test-649",\n      "gsm8k-test-736",\n      "gsm8k-test-1219",\n      "gsm8k-test-167",\n      "gsm8k-test-88",\n      "gsm8k-test-955",\n      "gsm8k-test-956",\n      "gsm8k-test-476",\n      "gsm8k-test-447",\n      "gsm8k-test-255",\n      "gsm8k-test-1043",\n      "gsm8k-test-130",\n      "gsm8k-test-607",\n      "gsm8k-test-378",\n      "gsm8k-test-61",\n      "gsm8k-test-1223",\n      "gsm8k-test-1298",\n      "gsm8k-test-652",\n      "gsm8k-test-1160",\n      "gsm8k-test-849",\n      "gsm8k-test-590",\n      "gsm8k-test-867",\n      "gsm8k-test-1010",\n      "gsm8k-test-375",\n      "gsm8k-test-1229",\n      "gsm8k-test-611",\n      "gsm8k-test-242",\n      "gsm8k-test-358",\n      "gsm8k-test-563",\n      "gsm8k-test-151",\n      "gsm8k-test-767",\n      "gsm8k-test-449",\n      "gsm8k-test-972",\n      "gsm8k-test-523",\n      "gsm8k-test-1170",\n      "gsm8k-test-695",\n      "gsm8k-test-552",\n      "gsm8k-test-283",\n      "gsm8k-test-1199",\n      "gsm8k-test-866",\n      "gsm8k-test-1105",\n      "gsm8k-test-321",\n      "gsm8k-test-138",\n      "gsm8k-test-161",\n      "gsm8k-test-507",\n      "gsm8k-test-273",\n      "gsm8k-test-638",\n      "gsm8k-test-291",\n      "gsm8k-test-929",\n      "gsm8k-test-158",\n      "gsm8k-test-1310",\n      "gsm8k-test-1074",\n      "gsm8k-test-200",\n      "gsm8k-test-481",\n      "gsm8k-test-1280",\n      "gsm8k-test-468",\n      "gsm8k-test-735",\n      "gsm8k-test-534",\n      "gsm8k-test-1",\n      "gsm8k-test-1078",\n      "gsm8k-test-683",\n      "gsm8k-test-626",\n      "gsm8k-test-131",\n      "gsm8k-test-204",\n      "gsm8k-test-542",\n      "gsm8k-test-1208",\n      "gsm8k-test-689",\n      "gsm8k-test-859",\n      "gsm8k-test-278",\n      "gsm8k-test-313",\n      "gsm8k-test-815",\n      "gsm8k-test-1104",\n      "gsm8k-test-1268",\n      "gsm8k-test-746",\n      "gsm8k-test-600",\n      "gsm8k-test-840",\n      "gsm8k-test-408",\n      "gsm8k-test-511",\n      "gsm8k-test-954",\n      "gsm8k-test-692",\n      "gsm8k-test-114",\n      "gsm8k-test-1294",\n      "gsm8k-test-924",\n      "gsm8k-test-817",\n      "gsm8k-test-351",\n      "gsm8k-test-6",\n      "gsm8k-test-182",\n      "gsm8k-test-517",\n      "gsm8k-test-1114",\n      "gsm8k-test-1311"\n    ],\n    "test": [\n      "gsm8k-test-1309",\n      "gsm8k-test-228",\n      "gsm8k-test-51",\n      "gsm8k-test-501",\n      "gsm8k-test-457",\n      "gsm8k-test-285",\n      "gsm8k-test-209",\n      "gsm8k-test-1116",\n      "gsm8k-test-178",\n      "gsm8k-test-1209",\n      "gsm8k-test-864",\n      "gsm8k-test-191",\n      "gsm8k-test-1034",\n      "gsm8k-test-1232",\n      "gsm8k-test-54",\n      "gsm8k-test-1149",\n      "gsm8k-test-407",\n      "gsm8k-test-451",\n      "gsm8k-test-919",\n      "gsm8k-test-1206",\n      "gsm8k-test-569",\n      "gsm8k-test-13",\n      "gsm8k-test-326",\n      "gsm8k-test-865",\n      "gsm8k-test-696",\n      "gsm8k-test-1291",\n      "gsm8k-test-318",\n      "gsm8k-test-440",\n      "gsm8k-test-189",\n      "gsm8k-test-778",\n      "gsm8k-test-198",\n      "gsm8k-test-704",\n      "gsm8k-test-1236",\n      "gsm8k-test-541",\n      "gsm8k-test-940",\n      "gsm8k-test-1098",\n      "gsm8k-test-775",\n      "gsm8k-test-1130",\n      "gsm8k-test-740",\n      "gsm8k-test-1182",\n      "gsm8k-test-393",\n      "gsm8k-test-142",\n      "gsm8k-test-466",\n      "gsm8k-test-592",\n      "gsm8k-test-163",\n      "gsm8k-test-1302",\n      "gsm8k-test-206",\n      "gsm8k-test-1286",\n      "gsm8k-test-928",\n      "gsm8k-test-747",\n      "gsm8k-test-333",\n      "gsm8k-test-758",\n      "gsm8k-test-727",\n      "gsm8k-test-429",\n      "gsm8k-test-146",\n      "gsm8k-test-350",\n      "gsm8k-test-1093",\n      "gsm8k-test-1314",\n      "gsm8k-test-334",\n      "gsm8k-test-946",\n      "gsm8k-test-777",\n      "gsm8k-test-1140",\n      "gsm8k-test-664",\n      "gsm8k-test-469",\n      "gsm8k-test-1306",\n      "gsm8k-test-646",\n      "gsm8k-test-821",\n      "gsm8k-test-548",\n      "gsm8k-test-135",\n      "gsm8k-test-432",\n      "gsm8k-test-1161",\n      "gsm8k-test-644",\n      "gsm8k-test-435",\n      "gsm8k-test-1022",\n      "gsm8k-test-810",\n      "gsm8k-test-939",\n      "gsm8k-test-292",\n      "gsm8k-test-1312",\n      "gsm8k-test-505",\n      "gsm8k-test-1103",\n      "gsm8k-test-538",\n      "gsm8k-test-1197",\n      "gsm8k-test-877",\n      "gsm8k-test-1195",\n      "gsm8k-test-741",\n      "gsm8k-test-1237",\n      "gsm8k-test-186",\n      "gsm8k-test-224",\n      "gsm8k-test-327",\n      "gsm8k-test-1307",\n      "gsm8k-test-788",\n      "gsm8k-test-781",\n      "gsm8k-test-958",\n      "gsm8k-test-1083",\n      "gsm8k-test-514",\n      "gsm8k-test-23",\n      "gsm8k-test-234",\n      "gsm8k-test-1099",\n      "gsm8k-test-1247",\n      "gsm8k-test-1287",\n      "gsm8k-test-1317",\n      "gsm8k-test-601",\n      "gsm8k-test-890",\n      "gsm8k-test-323",\n      "gsm8k-test-539",\n      "gsm8k-test-1025",\n      "gsm8k-test-365",\n      "gsm8k-test-217",\n      "gsm8k-test-1178",\n      "gsm8k-test-1297",\n      "gsm8k-test-1202",\n      "gsm8k-test-765",\n      "gsm8k-test-330",\n      "gsm8k-test-1086",\n      "gsm8k-test-663",\n      "gsm8k-test-1000",\n      "gsm8k-test-39",\n      "gsm8k-test-229",\n      "gsm8k-test-743",\n      "gsm8k-test-629",\n      "gsm8k-test-490",\n      "gsm8k-test-118",\n      "gsm8k-test-493",\n      "gsm8k-test-1269",\n      "gsm8k-test-175",\n      "gsm8k-test-995",\n      "gsm8k-test-141",\n      "gsm8k-test-1090",\n      "gsm8k-test-257",\n      "gsm8k-test-262",\n      "gsm8k-test-973",\n      "gsm8k-test-1125",\n      "gsm8k-test-338",\n      "gsm8k-test-1220",\n      "gsm8k-test-1080",\n      "gsm8k-test-433",\n      "gsm8k-test-411",\n      "gsm8k-test-1211",\n      "gsm8k-test-764",\n      "gsm8k-test-897",\n      "gsm8k-test-1059",\n      "gsm8k-test-247",\n      "gsm8k-test-460",\n      "gsm8k-test-43",\n      "gsm8k-test-471",\n      "gsm8k-test-14",\n      "gsm8k-test-145",\n      "gsm8k-test-120",\n      "gsm8k-test-64",\n      "gsm8k-test-676",\n      "gsm8k-test-1127",\n      "gsm8k-test-1052",\n      "gsm8k-test-487",\n      "gsm8k-test-570",\n      "gsm8k-test-994",\n      "gsm8k-test-438",\n      "gsm8k-test-1144",\n      "gsm8k-test-270",\n      "gsm8k-test-968",\n      "gsm8k-test-497",\n      "gsm8k-test-1113",\n      "gsm8k-test-833",\n      "gsm8k-test-389",\n      "gsm8k-test-193",\n      "gsm8k-test-1279",\n      "gsm8k-test-882",\n      "gsm8k-test-725",\n      "gsm8k-test-841",\n      "gsm8k-test-110",\n      "gsm8k-test-201",\n      "gsm8k-test-124",\n      "gsm8k-test-824",\n      "gsm8k-test-694",\n      "gsm8k-test-223",\n      "gsm8k-test-509",\n      "gsm8k-test-392",\n      "gsm8k-test-1109",\n      "gsm8k-test-918",\n      "gsm8k-test-287",\n      "gsm8k-test-1200",\n      "gsm8k-test-1118",\n      "gsm8k-test-947",\n      "gsm8k-test-154",\n      "gsm8k-test-907",\n      "gsm8k-test-103",\n      "gsm8k-test-30",\n      "gsm8k-test-1304",\n      "gsm8k-test-484",\n      "gsm8k-test-340",\n      "gsm8k-test-832",\n      "gsm8k-test-1117",\n      "gsm8k-test-985",\n      "gsm8k-test-437",\n      "gsm8k-test-1231",\n      "gsm8k-test-1126",\n      "gsm8k-test-337",\n      "gsm8k-test-776",\n      "gsm8k-test-4",\n      "gsm8k-test-799",\n      "gsm8k-test-543",\n      "gsm8k-test-931",\n      "gsm8k-test-584",\n      "gsm8k-test-1146",\n      "gsm8k-test-996",\n      "gsm8k-test-317",\n      "gsm8k-test-388",\n      "gsm8k-test-445",\n      "gsm8k-test-119",\n      "gsm8k-test-1190",\n      "gsm8k-test-642",\n      "gsm8k-test-117",\n      "gsm8k-test-102",\n      "gsm8k-test-976",\n      "gsm8k-test-1029",\n      "gsm8k-test-322",\n      "gsm8k-test-116",\n      "gsm8k-test-1040",\n      "gsm8k-test-164",\n      "gsm8k-test-380",\n      "gsm8k-test-140",\n      "gsm8k-test-139",\n      "gsm8k-test-826",\n      "gsm8k-test-245",\n      "gsm8k-test-504",\n      "gsm8k-test-81",\n      "gsm8k-test-858",\n      "gsm8k-test-647",\n      "gsm8k-test-418",\n      "gsm8k-test-643",\n      "gsm8k-test-488",\n      "gsm8k-test-1067",\n      "gsm8k-test-268",\n      "gsm8k-test-614",\n      "gsm8k-test-936",\n      "gsm8k-test-1037",\n      "gsm8k-test-148",\n      "gsm8k-test-19",\n      "gsm8k-test-938",\n      "gsm8k-test-75",\n      "gsm8k-test-550",\n      "gsm8k-test-218",\n      "gsm8k-test-518",\n      "gsm8k-test-271",\n      "gsm8k-test-357",\n      "gsm8k-test-902",\n      "gsm8k-test-70",\n      "gsm8k-test-900",\n      "gsm8k-test-250",\n      "gsm8k-test-1158",\n      "gsm8k-test-448",\n      "gsm8k-test-853",\n      "gsm8k-test-556",\n      "gsm8k-test-720",\n      "gsm8k-test-309",\n      "gsm8k-test-1207",\n      "gsm8k-test-669",\n      "gsm8k-test-1275",\n      "gsm8k-test-8",\n      "gsm8k-test-836",\n      "gsm8k-test-567",\n      "gsm8k-test-306",\n      "gsm8k-test-679",\n      "gsm8k-test-961",\n      "gsm8k-test-899",\n      "gsm8k-test-137",\n      "gsm8k-test-911",\n      "gsm8k-test-109",\n      "gsm8k-test-760",\n      "gsm8k-test-566",\n      "gsm8k-test-159",\n      "gsm8k-test-288",\n      "gsm8k-test-619",\n      "gsm8k-test-215",\n      "gsm8k-test-734",\n      "gsm8k-test-208",\n      "gsm8k-test-703",\n      "gsm8k-test-873",\n      "gsm8k-test-987",\n      "gsm8k-test-500",\n      "gsm8k-test-1153",\n      "gsm8k-test-927",\n      "gsm8k-test-1183",\n      "gsm8k-test-1064",\n      "gsm8k-test-52",\n      "gsm8k-test-94",\n      "gsm8k-test-1145",\n      "gsm8k-test-45",\n      "gsm8k-test-3",\n      "gsm8k-test-341",\n      "gsm8k-test-789",\n      "gsm8k-test-133",\n      "gsm8k-test-1030",\n      "gsm8k-test-165",\n      "gsm8k-test-759",\n      "gsm8k-test-452",\n      "gsm8k-test-564",\n      "gsm8k-test-722",\n      "gsm8k-test-574",\n      "gsm8k-test-9",\n      "gsm8k-test-1235",\n      "gsm8k-test-904",\n      "gsm8k-test-925",\n      "gsm8k-test-152",\n      "gsm8k-test-558",\n      "gsm8k-test-36",\n      "gsm8k-test-854",\n      "gsm8k-test-1292",\n      "gsm8k-test-596",\n      "gsm8k-test-565",\n      "gsm8k-test-1284",\n      "gsm8k-test-42",\n      "gsm8k-test-315",\n      "gsm8k-test-373",\n      "gsm8k-test-920",\n      "gsm8k-test-881",\n      "gsm8k-test-40",\n      "gsm8k-test-1137",\n      "gsm8k-test-366",\n      "gsm8k-test-977",\n      "gsm8k-test-698",\n      "gsm8k-test-1271",\n      "gsm8k-test-682",\n      "gsm8k-test-105",\n      "gsm8k-test-362",\n      "gsm8k-test-798",\n      "gsm8k-test-573",\n      "gsm8k-test-905",\n      "gsm8k-test-895",\n      "gsm8k-test-416",\n      "gsm8k-test-635",\n      "gsm8k-test-885",\n      "gsm8k-test-166",\n      "gsm8k-test-819",\n      "gsm8k-test-830",\n      "gsm8k-test-181",\n      "gsm8k-test-422",\n      "gsm8k-test-25",\n      "gsm8k-test-183",\n      "gsm8k-test-754",\n      "gsm8k-test-801",\n      "gsm8k-test-421",\n      "gsm8k-test-1073",\n      "gsm8k-test-685",\n      "gsm8k-test-884",\n      "gsm8k-test-752",\n      "gsm8k-test-1012",\n      "gsm8k-test-254",\n      "gsm8k-test-1258",\n      "gsm8k-test-806",\n      "gsm8k-test-718",\n      "gsm8k-test-1101",\n      "gsm8k-test-391",\n      "gsm8k-test-1165",\n      "gsm8k-test-1044",\n      "gsm8k-test-227",\n      "gsm8k-test-1023",\n      "gsm8k-test-1063",\n      "gsm8k-test-312",\n      "gsm8k-test-814",\n      "gsm8k-test-233",\n      "gsm8k-test-1187",\n      "gsm8k-test-24",\n      "gsm8k-test-675",\n      "gsm8k-test-197",\n      "gsm8k-test-336",\n      "gsm8k-test-71",\n      "gsm8k-test-791",\n      "gsm8k-test-862",\n      "gsm8k-test-359",\n      "gsm8k-test-656",\n      "gsm8k-test-521",\n      "gsm8k-test-409",\n      "gsm8k-test-549",\n      "gsm8k-test-339",\n      "gsm8k-test-28",\n      "gsm8k-test-986",\n      "gsm8k-test-267",\n      "gsm8k-test-594",\n      "gsm8k-test-1018",\n      "gsm8k-test-879",\n      "gsm8k-test-111",\n      "gsm8k-test-610",\n      "gsm8k-test-444",\n      "gsm8k-test-353",\n      "gsm8k-test-805",\n      "gsm8k-test-446",\n      "gsm8k-test-620",\n      "gsm8k-test-850",\n      "gsm8k-test-394",\n      "gsm8k-test-194",\n      "gsm8k-test-260",\n      "gsm8k-test-957",\n      "gsm8k-test-837",\n      "gsm8k-test-1168",\n      "gsm8k-test-532",\n      "gsm8k-test-551",\n      "gsm8k-test-1151",\n      "gsm8k-test-949",\n      "gsm8k-test-909",\n      "gsm8k-test-686"\n    ]\n  },\n  "hotpotqa": {\n    "dev": [\n      "5abe34095542993f32c2a088",\n      "5a8b3de95542997f31a41cc3",\n      "5a7557d75542992d0ec05f68",\n      "5ab99a215542996be202056a",\n      "5a7f567e5542992097ad2f22",\n      "5a84e9d35542994c784dda84",\n      "5ab4b9f75542990594ba9ca4",\n      "5abe3dc65542993f32c2a0c4",\n      "5ab4355555429942dd415ee3",\n      "5ab6087a554299110f2199be",\n      "5ab5ec0a5542997d4ad1f250",\n      "5a74f63e5542993748c89758",\n      "5ab4774d5542991751b4d781",\n      "5ae67dba55429908198fa5f0",\n      "5ac54c435542993e66e822d1",\n      "5a84f5195542994c784dda9d",\n      "5a8319b255429940e5e1a960",\n      "5ae129115542990adbacf722",\n      "5ac3d9135542995c82c4ac4c",\n      "5ae361805542992e3233c3c9",\n      "5a840e395542992ef85e239d",\n      "5a72dcb45542992359bc31af",\n      "5a8aeadc55429950cd6afbe0",\n      "5ab972485542996be20204f6",\n      "5a8d12ca5542994ba4e3dbe2",\n      "5ab2acbc55429929539467d2",\n      "5a875fee5542996e4f3087ab",\n      "5abb4b4a5542993f40c73aea",\n      "5ab827a555429934fafe6cf5",\n      "5ab295eb554299194fa93412",\n      "5a82fcbd55429954d2e2ebe8",\n      "5a809e9f5542996402f6a5b1",\n      "5ae8167b55429952e35eaa06",\n      "5a8b50475542997f31a41cd4",\n      "5abe765a55429976d4830b1f",\n      "5adbcc085542996e6852523c",\n      "5a8a7d2b55429930ff3c0dfc",\n      "5abb8a9b5542996cc5e4a016",\n      "5ae518655542993aec5ec139",\n      "5a8638945542994775f6072d",\n      "5a77c74155429967ab105291",\n      "5a8ed10b5542995085b374a0",\n      "5a75f0ea5542994ccc91866c",\n      "5abe953b5542993f32c2a170",\n      "5ade4fcc5542997c77adeded",\n      "5a721e0e55429971e9dc928d",\n      "5a7e05ba5542995f4f402392",\n      "5a7be1ab5542997c3ec972a9",\n      "5ab7b0775542992aa3b8c83e",\n      "5ae0ae4555429945ae959419",\n      "5a7f22005542993067513656",\n      "5a802b595542995d8a8ddf51",\n      "5a7e42df5542994959419954",\n      "5a7ce0ce554299452d57ba92",\n      "5abe87cb5542993f32c2a156",\n      "5ac009bf5542997d642959a7",\n      "5a72abb95542992359bc3162",\n      "5a778a8e5542992a6e59deca",\n      "5a8ac4dd5542992d82986f76",\n      "5ab6c47e55429954757d3368",\n      "5adcdf455542994ed6169c11",\n      "5ab29c24554299449642c932",\n      "5a7562045542992db947364f",\n      "5a9030ef5542995651fb50e4",\n      "5addcb215542997dc7907033",\n      "5ae524415542992663a4f121",\n      "5ab90d8955429916710eb0f1",\n      "5a89bbb05542992e4fca83a3",\n      "5a721a7655429971e9dc9271",\n      "5a7c0dab5542997c3ec972d2",\n      "5abfa2255542993fe9a41e18",\n      "5a8a833855429970aeb702fe",\n      "5a7af2a455429931da12c998",\n      "5a80ecea55429938b6142254",\n      "5ae14b5c55429920d52343aa",\n      "5a7137dc5542994082a3e68a",\n      "5a7e618155429949594199b0",\n      "5ac35a7e554299741d48a257",\n      "5ab849d955429916710eb017",\n      "5a851b0d5542997175ce1f7f",\n      "5a7fc8d05542992e7d278d86",\n      "5add0f635542994ed6169c56",\n      "5ae7eb43554299540e5a5695",\n      "5a849c135542992a431d1a7a",\n      "5a877dd65542993e715abf79",\n      "5a80ddc555429938b6142224",\n      "5adfad2b554299603e41835c",\n      "5ab90d2755429916710eb0f0",\n      "5ab707d65542991d32223761",\n      "5a7af23a55429931da12c994",\n      "5a86ab7755429960ec39b6bf",\n      "5ae127c9554299422ee99612",\n      "5ae5e62455429929b0807a07",\n      "5ae35e8c5542990afbd1e120",\n      "5a8d9de2554299068b959d62",\n      "5a8f503c5542992414482a34",\n      "5ae2d8fb55429928c42394fa",\n      "5abace6b5542996cc5e49e99",\n      "5a7cc59c55429909bec767fb",\n      "5ab916e155429919ba4e23a4"\n    ],\n    "test": [\n      "5ae143ed55429920d5234360",\n      "5abc19705542993a06baf86e",\n      "5ac3e0f7554299194317388b",\n      "5ab985eb554299131ca42360",\n      "5a8753d95542994846c1cd63",\n      "5ab97d0a5542996be202051e",\n      "5adef1b35542993a75d263af",\n      "5a73b55855429978a71e9086",\n      "5ae80919554299540e5a56f6",\n      "5ae361605542994393b9e69b",\n      "5ac3a76e554299741d48a2be",\n      "5ab7f6d35542993667794070",\n      "5a86769c5542994775f60776",\n      "5ab31864554299233954ff06",\n      "5a7b1023554299042af8f6c2",\n      "5ac19f405542991316484b5b",\n      "5ae382ee5542992e3233c430",\n      "5ab2659e554299340b5254b2",\n      "5adcbb815542994ed6169bde",\n      "5a74f8bd5542993748c8976a",\n      "5ade25ed5542997c77aded70",\n      "5a78ae8b5542990784727730",\n      "5ae12a4b55429920d52342df",\n      "5ae22eae554299495565da30",\n      "5ab6aa4c55429953192ad359",\n      "5adf3e835542993a75d26442",\n      "5a8ee3a755429917b4a5be02",\n      "5ac0dcf25542996f0d89cc2c",\n      "5a713b325542994082a3e6b5",\n      "5a87c13f5542996e4f30890c",\n      "5a8843cd5542994846c1ce75",\n      "5ae788fa55429952e35ea964",\n      "5ae1925a554299492dc91b48",\n      "5ae1443855429920d5234362",\n      "5a7a77425542995eb53be83e",\n      "5a7153d05542994082a3e7dc",\n      "5a72688c5542997f827839b2",\n      "5a83411655429966c78a6b5d",\n      "5adbd70c55429947ff173843",\n      "5a7796e05542992a6e59df0f",\n      "5abc030e554299642a094bdc",\n      "5a7bb7f2554299042af8f7b6",\n      "5ab3bfd3554299233954ff99",\n      "5ab9b7d555429970cfb8eb7a",\n      "5ab6bf6d55429953192ad372",\n      "5ae7cb9b5542994a481bbde2",\n      "5ab1d6545542997061209588",\n      "5ae528ed5542993aec5ec16e",\n      "5a8c7ced5542995e66a4761c",\n      "5a72f74a55429901807daf59",\n      "5ae1d4e8554299234fd04312",\n      "5a80aeef5542992097ad301a",\n      "5a7f66f155429969796c1a33",\n      "5abbda84554299642a094b5b",\n      "5ae489ab5542995ad6573d62",\n      "5a7e1b7155429965cec5ea6c",\n      "5a8fae205542997ba9cb32ec",\n      "5ae588cb55429960a22e0300",\n      "5a75e62f5542992d0ec06013",\n      "5ac113e2554299294b21909a",\n      "5a82fe5055429966c78a6ad3",\n      "5a7264ed5542990c210a411a",\n      "5add8f055542990dbb2f7e6d",\n      "5a7a112a5542990783324e2b",\n      "5ab870ee5542990e739ec8f6",\n      "5a8810485542997e5c09a590",\n      "5a8208ee5542990a1d231f1e",\n      "5a86399e5542994775f60733",\n      "5a78e9a2554299148911f995",\n      "5a84995d5542997b5ce3fee4",\n      "5ab941c2554299743d22ea88",\n      "5a8e5f1f5542995a26add4d6",\n      "5abf1ad15542997ec76fd3c2",\n      "5ae5cf625542996de7b71a22",\n      "5a7c8ae855429935c91b5207",\n      "5a87106555429960ec39b71f",\n      "5a7f296f55429934daa2fd10",\n      "5a88b3b4554299206df2b336",\n      "5add67705542992ae4cec54f",\n      "5ab2b630554299545a2cfa17",\n      "5a851ba95542997175ce1f81",\n      "5ae1d0e8554299234fd0430a",\n      "5ae1ebf95542997283cd22f2",\n      "5a81d0515542990a1d231ed4",\n      "5a87ed5b5542994846c1cdfc",\n      "5abc41315542993a06baf8bb",\n      "5a776a805542993735360269",\n      "5a78e3b955429974737f78f1",\n      "5a78b5f7554299148911f95b",\n      "5ab4304a55429942dd415ec5",\n      "5a7e0ed25542991319bc9424",\n      "5ab9e477554299232ef4a25d",\n      "5ab900045542991b5579f0c0",\n      "5ab6e848554299710c8d1faa",\n      "5a7dc2495542997cc2c4749a",\n      "5a7cfea2554299452d57babe",\n      "5ae497f15542995ad6573db8",\n      "5ae2133e554299495565d9e3",\n      "5abfe1115542997d64295909",\n      "5a8ce55f554299441c6b9f62",\n      "5a7c47c35542990527d55481",\n      "5ab4107a554299753aec5a2f",\n      "5a774e415542994aec3b7280",\n      "5abdf06355429976d4830a2d",\n      "5a8ec55f5542995a26add50c",\n      "5ae1338d55429901ffe4ade1",\n      "5a7bea3d5542996dd594b847",\n      "5ac26ac15542992f1f2b38bc",\n      "5ae531695542990ba0bbb1fb",\n      "5ac2cb09554299657fa29066",\n      "5a82817555429954d2e2eb5a",\n      "5ae34b275542992e3233c371",\n      "5a74f5155542993748c89750",\n      "5ab4428c5542990594ba9bda",\n      "5ae532365542992663a4f150",\n      "5a72f1cf5542994cef4bc414",\n      "5a749e1855429979e28829c3",\n      "5a8b051255429971feec460e",\n      "5a7616755542992d0ec06028",\n      "5ac0fd56554299012d1db679",\n      "5adde73f5542992200553b94",\n      "5ab506a05542990594ba9cd6",\n      "5a7f57a45542992097ad2f29",\n      "5a8dee2455429917b4a5bce1",\n      "5a7cd7f055429909bec7683e",\n      "5a7cb8bc554299452d57b9be",\n      "5ab9547755429970cfb8ea6c",\n      "5a8cfa2e554299585d9e378b",\n      "5ab67bd355429954757d32f2",\n      "5abecdf45542997719eab5d4",\n      "5a7e37095542995ed0d166d5",\n      "5a8f18d555429918e830d189",\n      "5a7b85ac5542997c3ec971ef",\n      "5ae617285542996de7b71b4c",\n      "5ae7247b5542992ae0d163c1",\n      "5ae306115542992decbdcdc4",\n      "5abdd08d5542991f6610604c",\n      "5a7f54665542992097ad2f1a",\n      "5ae0bc185542993d6555ec29",\n      "5a864d9e5542994775f6073d",\n      "5ae5b25d5542990ba0bbb2bb",\n      "5ae0120155429925eb1afbfb",\n      "5ae2b770554299495565db0f",\n      "5ab8634e5542990e739ec8de",\n      "5a8836445542997e5c09a5a5",\n      "5ac194db55429964131be23a",\n      "5ab8eabb5542991b5579f03d",\n      "5ab234c65542993be8fa98c6",\n      "5abffc0d5542990832d3a1e2",\n      "5ae6914755429908198fa627",\n      "5ae78f3b554299540e5a5608",\n      "5a8b46925542995d1e6f133c",\n      "5ab4f7fc5542991779162d43",\n      "5abd13b655429924427fceea",\n      "5adf37a95542995ec70e8f97",\n      "5a78e86f55429974737f78fb",\n      "5a807ceb554299485f598616",\n      "5a8c493e554299653c1aa020",\n      "5a75404a55429916b01642be",\n      "5a81108855429926c1cdace2",\n      "5ae340a85542994393b9e668",\n      "5a8d0be85542994ba4e3dbb2",\n      "5add43335542995b365faae1",\n      "5ae4d8475542993aec5ec0e0",\n      "5ab5c263554299488d4d9a18",\n      "5a74a26a55429979e28829d6",\n      "5a8d841955429941ae14dfe6",\n      "5ac2d85e55429921a00ab06b",\n      "5a7aa0a55542990198eaf165",\n      "5ab9ddef5542994dbf019875",\n      "5ae157ed5542990adbacf75d",\n      "5a8555ea5542992a431d1b07",\n      "5ae141a65542997b2ef7d18a",\n      "5ae4a1ef55429970de88d9e7",\n      "5ab90e9755429934fafe6e51",\n      "5a8e3ea95542995a26add48d",\n      "5ac2f34455429921a00ab0b6",\n      "5a7344e95542991f9a20c6ce",\n      "5a7b1c6b55429931da12c9ca",\n      "5ab341f755429969a97a8114",\n      "5a8cddd1554299441c6b9f3a",\n      "5a88f7855542993b751ca8da",\n      "5a7300b85542994cef4bc41a",\n      "5ae7f56e5542993210983fd1",\n      "5adda7585542997dc790700c",\n      "5ab4314955429942dd415ecd",\n      "5a81a2705542990a1d231e30",\n      "5ae291bb5542996483e649f6",\n      "5ab207f65542993be8fa98b8",\n      "5ae152a65542990adbacf74f",\n      "5a739ef955429978a71e904b",\n      "5ab2b61d5542991669774065",\n      "5a88fc4255429938390d4019",\n      "5ae2c416554299492dc91c67",\n      "5a7a51b95542990783324f0f",\n      "5ae3006c55429928c423959f",\n      "5a74122f55429929fddd83da",\n      "5a81711455429938b614233e",\n      "5ae1e4095542997283cd22bf",\n      "5a8c6d4e5542996e8ac88aa6",\n      "5abf11d45542997719eab660",\n      "5ac11201554299294b21908c",\n      "5ade063d5542995b365fabcd",\n      "5ab3a83d5542992ade7c6e08",\n      "5a79b525554299029c4b5f67",\n      "5a8b77705542995d1e6f13aa",\n      "5abd0296554299114383a1d1",\n      "5ae4d0ff55429913cc204511",\n      "5ac07fff554299294b219006",\n      "5ab741005542992aa3b8c800",\n      "5abe4a365542993f32c2a0dc",\n      "5ab6cebd55429954757d3378",\n      "5abc219655429959677d6a1d",\n      "5ac546975542993e66e8229a",\n      "5adf4da15542995534e8c780",\n      "5a7f1d3c5542993067513639",\n      "5ae3c71f5542991a06ce9a22",\n      "5a7e32905542991319bc943b",\n      "5a75462d5542996c70cfaed3",\n      "5a8effb55542997ba9cb317f",\n      "5abec66b5542997ec76fd360",\n      "5a762cf0554299109176e6a4",\n      "5ade1c9e55429939a52fe822",\n      "5abb1e1b5542992ccd8e7f19",\n      "5a791ce955429974737f798e",\n      "5a7ef6b35542994959419a94",\n      "5ae5e8dc554299546bf82fbe",\n      "5adcf8155542990d50227d6a",\n      "5a74feb75542996c70cfae6d",\n      "5ae32e125542991a06ce9946",\n      "5a8242c155429940e5e1a81d",\n      "5abd7d0c55429924427fd010",\n      "5ae6860e5542991bbc976112",\n      "5a849cc95542997175ce1efa",\n      "5ac1a04e5542991316484b63",\n      "5ade7a435542997c77adee3a",\n      "5a865e8a55429960ec39b67a",\n      "5adc0e0355429944faac2404",\n      "5ae7cea355429952e35ea9c1",\n      "5a77a44d5542997042120aba",\n      "5a775f8a5542994aec3b72ba",\n      "5a80870d5542992bc0c4a752",\n      "5ae4d7fc5542990ba0bbb16f",\n      "5ab431e65542991751b4d6e2",\n      "5ae71f06554299572ea54708",\n      "5abdce1b5542993062266cfe",\n      "5a8c5eca554299240d9c213d",\n      "5a8c8f24554299653c1aa0ba",\n      "5ab285d1554299449642c8dd",\n      "5a78e49855429974737f78f3",\n      "5a72b3335542994cef4bc3dc",\n      "5a82bc7455429940e5e1a926",\n      "5a77b0395542997042120ae1",\n      "5a90b6985542990a984936c2",\n      "5a7fd95d5542992e7d278dc5",\n      "5ab1e36d554299449642c7f9",\n      "5adfff0755429925eb1afbce",\n      "5ab5e024554299494045f097",\n      "5a810506554299260e20a202",\n      "5a71231a5542994082a3e5c4",\n      "5a71717d5542994082a3e834",\n      "5a74adb455429916b01641c6",\n      "5ab28c89554299340b525513",\n      "5ab626d555429953192ad279",\n      "5ab3589455429976abd1bc93",\n      "5a8af1e255429950cd6afc28",\n      "5ac11ee45542997d64295a7d",\n      "5ae520315542993aec5ec152",\n      "5a8f68795542992414482ab5",\n      "5ae37e525542992e3233c42d",\n      "5ade51515542992fa25da745",\n      "5ade02d05542997545bbbe0d",\n      "5ac41850554299194317390b",\n      "5a727e1b5542991f9a20c497",\n      "5a827ce155429940e5e1a8d7",\n      "5ae5415055429908b63265b7",\n      "5ae20427554299234fd04397",\n      "5adcd7325542994d58a2f6f6",\n      "5ab64cfa55429953192ad2b7",\n      "5ae4bb1255429913cc2044e7",\n      "5abe2b0155429976d4830a83",\n      "5ae0d52e554299603e418469",\n      "5abc9fb055429959677d6ac9",\n      "5ac083b0554299294b21900d",\n      "5a8b3e985542995d1e6f131c",\n      "5a7c6b6655429935c91b519a",\n      "5a82d71555429966c78a6a94",\n      "5abb7ea4554299642a094a96",\n      "5a7f00b455429930675135f1",\n      "5ae153915542990adbacf752",\n      "5a7bf3da5542996dd594b869",\n      "5ae1f2f3554299234fd0435a",\n      "5a8f0796554299458435d528",\n      "5ae81b895542997ec272770c",\n      "5ae0f6f5554299422ee99583",\n      "5a7c6d98554299683c1c6304",\n      "5a83de04554299334474609f",\n      "5ae20c245542997283cd2367",\n      "5ac4efa85542991943173a04",\n      "5ac385f8554299741d48a279",\n      "5add23d05542994734353871",\n      "5add15d15542994ed6169c70",\n      "5ae5d82f554299546bf82f9a",\n      "5ab29b07554299194fa93440",\n      "5a88bedb554299206df2b35a",\n      "5abfd3ba5542994516f4550a",\n      "5a728b1c5542992359bc30e0",\n      "5a806aca5542996402f6a503",\n      "5ae4bbf05542995ad6573e32",\n      "5ab3bcc55542992ade7c6e66",\n      "5abd68b45542993062266c77",\n      "5ac2a399554299218029dada",\n      "5a8f2fed55429918e830d1c7",\n      "5a80abb65542996402f6a5df",\n      "5a8f45ca5542997ba9cb320e",\n      "5a87aa135542996e4f3088b8",\n      "5ab29346554299545a2cf997",\n      "5ae789615542997ec2727695",\n      "5ac45b58554299076e296dad",\n      "5ac43ddf5542995c82c4ad08",\n      "5ae075455542993d6555eb59",\n      "5a7350585542992359bc32f9",\n      "5ae4b55455429913cc2044db",\n      "5a8494b15542991dd0999d35",\n      "5abf16965542993fe9a41dd3",\n      "5ae60426554299546bf83019",\n      "5ae226755542996483e64906",\n      "5a809b575542996402f6a59c",\n      "5ae28058554299495565da90",\n      "5ae7ff745542994a481bbe6e",\n      "5a7bbc59554299294a54aaa9",\n      "5a89f58f5542992e4fca84af",\n      "5ae5db98554299546bf82fa6",\n      "5ac160bb5542994d76dccdfb",\n      "5a7a33205542996a35c1712f",\n      "5a89df265542992e4fca840d",\n      "5ae3dfb05542992e3233c490",\n      "5a837121554299123d8c2128",\n      "5a77b1a75542992a6e59df8d",\n      "5ade15f35542997545bbbe47",\n      "5ac5632e5542993e66e823d9",\n      "5ab72b5c5542993667793f63",\n      "5ae4c35c55429913cc2044fb",\n      "5a8284b555429954d2e2eb5f",\n      "5a777ca855429949eeb29e6a",\n      "5ae1dac95542997283cd229b",\n      "5a808f3f5542992097ad2ffd",\n      "5ae6a5d15542991bbc97615d",\n      "5ab8111c5542991d3222382b",\n      "5ab3c7df55429969a97a81be",\n      "5ae682c45542995703ce8b6f",\n      "5a7b542455429927d897bf92",\n      "5ade00855542997dc7907118",\n      "5adfd0d455429942ec259b4d",\n      "5a7615fb5542992db9473727",\n      "5a8747915542994775f607bb",\n      "5a8b34705542995d1e6f130d",\n      "5ab5ac345542992aa134a37f",\n      "5ab7f7465542993667794071",\n      "5a8979f4554299669944a52e",\n      "5a8bd2965542997f31a41dd1",\n      "5a8da2b05542994ba4e3dcdc",\n      "5ac4f9db55429924173fb532",\n      "5a8cee9b554299441c6b9f87",\n      "5a8220db5542995ce29dccc1",\n      "5add56ad5542992200553abf",\n      "5ab44ab25542991779162c58",\n      "5a8b057255429971feec4611",\n      "5ae4be5c5542995ad6573e3a",\n      "5abaf4535542996cc5e49f35",\n      "5adefcf15542995ec70e8f4c",\n      "5a883afc5542994846c1ce48",\n      "5ab66e1355429954757d32b2",\n      "5a7fa08d55429969796c1af7",\n      "5ae8202255429952e35eaa3a",\n      "5ab57759554299637185c5a5",\n      "5ab96c93554299743d22eb63",\n      "5ae2e61455429928c423953e",\n      "5ac0aa595542992a796ded5a",\n      "5abcf46255429959677d6b6b",\n      "5ac2ab9455429967731025f0",\n      "5a8f4c8d554299458435d5a3",\n      "5a7717635542994aec3b71e3",\n      "5a89019a5542995153361259",\n      "5addf3d65542990dbb2f7f0e",\n      "5a7e5ec25542991319bc947a",\n      "5a7dbc175542995ed0d1666b",\n      "5a7521c55542993748c897c6",\n      "5a8591715542994c784ddb27",\n      "5abce7d755429959677d6b3a",\n      "5ab9535a5542996be202047e",\n      "5ae1faf15542997f29b3c1e3",\n      "5a8a231a55429930ff3c0cb4",\n      "5ae37d265542994393b9e6e7",\n      "5a8e83545542990e94052b28",\n      "5ac5188255429924173fb5a3",\n      "5a7351785542994cef4bc539",\n      "5a8a03245542992e4fca84d5",\n      "5abf8e135542990832d3a14f",\n      "5ab5d6155542997d4ad1f1fa"\n    ]\n  }\n}')

# Write pre-split local gsm8k dataset
with open("cgsc/data/gsm8k.jsonl", "wb") as f:
    f.write(gzip.decompress(base64.b64decode('...'.encode("ascii"))))

with open("cgsc/run.py", "w", encoding="utf-8") as f:
    f.write('"""CLI entry point for running CGSC pipeline."""\n\nimport argparse\nimport os\nimport sys\nimport time\nimport json\nimport re\nimport math\nimport zipfile\nfrom transformers import AutoTokenizer\nfrom vllm import LLM, SamplingParams\n\nfrom cgsc.scoring import (\n    extract_final_answer,\n    score_gsm8k,\n    score_hotpotqa,\n    majority_vote,\n    exact_match_score,\n)\nfrom cgsc.sanity import run_sanity_checks\n\n\ndef clean_prompt(text: str) -> str:\n    lines = text.splitlines()\n    filtered = [l for l in lines if not l.strip().startswith("#")]\n    return "\\n".join(filtered).strip("\\n")\n\n\ndef load_all_prompts(prompts_dir: str):\n    prompts = {}\n    files = {\n        "a0_gsm8k": "a0_gsm8k.txt",\n        "a0_hotpotqa": "a0_hotpotqa.txt",\n        "verb": "verb.txt",\n        "verif_gsm8k": "verif_gsm8k.txt",\n        "verif_hotpotqa": "verif_hotpotqa.txt",\n        "rev": "revision.txt",\n        "ioe": "ioe.txt",\n    }\n    for k, fname in files.items():\n        p = os.path.join(prompts_dir, fname)\n        with open(p, "r", encoding="utf-8") as f:\n            prompts[k] = clean_prompt(f.read())\n    return prompts\n\n\ndef get_dataset_records(dataset_name: str):\n    local_path = os.path.join("cgsc/data", f"{dataset_name}.jsonl")\n    if os.path.exists(local_path):\n        print(f"Loading {dataset_name} records from local file {local_path}...")\n        records = {}\n        with open(local_path, "r", encoding="utf-8") as f:\n            for line in f:\n                if line.strip():\n                    item = json.loads(line)\n                    records[item["id"]] = {\n                        "id": item["id"],\n                        "question": item["question"].strip(),\n                        "gold": str(item["gold"]).strip(),\n                        "context": item.get("context", ""),\n                    }\n        print(f"Loaded {len(records)} records from {local_path}.")\n        return records\n\n    # Fallback to Hugging Face datasets if local file is missing\n    print(f"Local {local_path} not found; falling back to Hugging Face datasets...")\n    from datasets import load_dataset\n    if dataset_name == "gsm8k":\n        ds = load_dataset("openai/gsm8k", "main", split="test")\n        records = {}\n        for idx, item in enumerate(ds):\n            gold = item["answer"].split("####")[-1].strip()\n            records[f"gsm8k-test-{idx}"] = {\n                "id": f"gsm8k-test-{idx}",\n                "question": item["question"].strip(),\n                "gold": gold,\n                "context": "",\n            }\n        return records\n    elif dataset_name == "hotpotqa":\n        ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")\n        records = {}\n        for idx, item in enumerate(ds):\n            ctx_titles = item["context"]["title"]\n            ctx_sents = item["context"]["sentences"]\n            paras = []\n            for t, s_list in zip(ctx_titles, ctx_sents):\n                paras.append(f"{t}: {\' \'.join(s_list)}")\n            context_str = "\\n\\n".join(paras)\n            qid = str(item["id"]).strip()\n            records[qid] = {\n                "id": qid,\n                "question": item["question"].strip(),\n                "gold": item["answer"].strip(),\n                "context": context_str,\n            }\n        return records\n    else:\n        raise ValueError(f"Unknown dataset {dataset_name}")\n\n\ndef load_model(model_key: str):\n    if model_key == "qwen":\n        model_id = "Qwen/Qwen2.5-7B-Instruct"\n    elif model_key == "llama":\n        model_id = "meta-llama/Llama-3.1-8B-Instruct"\n    else:\n        raise ValueError(f"Unknown model key {model_key}")\n\n    print(f"\\nLoading tokenizer for {model_id}...")\n    sys.stdout.flush()\n    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)\n\n    import torch\n\n    compute_cap = (\n        torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 8\n    )\n    model_dtype = "bfloat16" if compute_cap >= 8 else "float16"\n    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1\n    print(\n        f"GPUs: {num_gpus}, compute_cap: {compute_cap}, chosen dtype: {model_dtype}"\n    )\n\n    print(f"Loading vLLM model: {model_id}...")\n    sys.stdout.flush()\n    llm = LLM(\n        model=model_id,\n        tensor_parallel_size=num_gpus,\n        max_model_len=8192,\n        enable_prefix_caching=True,\n        dtype=model_dtype,\n        gpu_memory_utilization=0.88,\n        trust_remote_code=True,\n    )\n    print("Model loaded successfully into vLLM!\\n")\n    sys.stdout.flush()\n    return llm, tokenizer\n\n\ndef run_batch_pipeline(\n    records_to_run, model_key, dataset_key, llm, tokenizer, prompts\n):\n    results = []\n    start_total_time = time.time()\n    n_items = len(records_to_run)\n    print(\n        f"Starting batch pipeline for {n_items} records ({model_key}/{dataset_key})..."\n    )\n    sys.stdout.flush()\n\n    sp_greedy_768 = SamplingParams(temperature=0.0, max_tokens=768)\n    sp_verb = SamplingParams(temperature=0.0, max_tokens=8)\n    sp_verif = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)\n\n    # 1. Generate A0\n    print("Generating A0...")\n    sys.stdout.flush()\n    a0_prompts = []\n    for r in records_to_run:\n        if dataset_key == "gsm8k":\n            user_msg = prompts["a0_gsm8k"].format(question=r["question"])\n        else:\n            user_msg = prompts["a0_hotpotqa"].format(\n                context=r["context"], question=r["question"]\n            )\n        conv = [{"role": "user", "content": user_msg}]\n        text_prompt = tokenizer.apply_chat_template(\n            conv, tokenize=False, add_generation_prompt=True\n        )\n        a0_prompts.append((r, user_msg, text_prompt))\n\n    a0_outputs = llm.generate([p[2] for p in a0_prompts], sp_greedy_768)\n\n    verb_prompts = []\n    sc_prompts = []\n    verif_prompts = []\n    rev_prompts = []\n    ioe_prompts = []\n\n    for (r, user_msg, text_prompt), out in zip(a0_prompts, a0_outputs):\n        a0_text = out.outputs[0].text\n        a0_extracted = extract_final_answer(a0_text)\n\n        conv_verb = [\n            {"role": "user", "content": user_msg},\n            {"role": "assistant", "content": a0_text},\n            {"role": "user", "content": prompts["verb"]},\n        ]\n        verb_prompts.append(\n            tokenizer.apply_chat_template(\n                conv_verb, tokenize=False, add_generation_prompt=True\n            )\n        )\n        sc_prompts.append(text_prompt)\n\n        if dataset_key == "gsm8k":\n            verif_user_msg = prompts["verif_gsm8k"].format(\n                question=r["question"], extracted_A0_answer=a0_extracted\n            )\n        else:\n            verif_user_msg = prompts["verif_hotpotqa"].format(\n                question=r["question"],\n                context=r["context"],\n                extracted_A0_answer=a0_extracted,\n            )\n        conv_verif = [{"role": "user", "content": verif_user_msg}]\n        verif_base = tokenizer.apply_chat_template(\n            conv_verif, tokenize=False, add_generation_prompt=True\n        )\n        verif_prompt_full = verif_base + "The proposed answer is: ("\n        verif_prompts.append((verif_user_msg, a0_extracted, verif_prompt_full))\n\n        conv_rev = [\n            {"role": "user", "content": user_msg},\n            {"role": "assistant", "content": a0_text},\n            {"role": "user", "content": prompts["rev"]},\n        ]\n        rev_prompts.append(\n            tokenizer.apply_chat_template(\n                conv_rev, tokenize=False, add_generation_prompt=True\n            )\n        )\n\n        conv_ioe = [\n            {"role": "user", "content": user_msg},\n            {"role": "assistant", "content": a0_text},\n            {"role": "user", "content": prompts["ioe"]},\n        ]\n        ioe_prompts.append(\n            tokenizer.apply_chat_template(\n                conv_ioe, tokenize=False, add_generation_prompt=True\n            )\n        )\n\n    print("Generating S_verb...")\n    sys.stdout.flush()\n    verb_outputs = llm.generate(verb_prompts, sp_verb)\n\n    # Generate the 10 self-consistency samples as 10 separate requests with seeds 42, 43, ..., 51\n    print("Generating S_sc (10 separate samples with seeds 42..51)...")\n    sys.stdout.flush()\n    sc_sample_outputs = []  # shape: (10, n_items)\n    for j in range(10):\n        sp_j = SamplingParams(\n            n=1,\n            temperature=0.7,\n            top_p=0.95,\n            seed=42 + j,\n            max_tokens=768,\n        )\n        out_j = llm.generate(sc_prompts, sp_j)\n        sc_sample_outputs.append(out_j)\n\n    print("Generating S_verif (P(True))...")\n    sys.stdout.flush()\n    verif_outputs = llm.generate([vp[2] for vp in verif_prompts], sp_verif)\n\n    print("Generating A1 (revision)...")\n    sys.stdout.flush()\n    rev_outputs = llm.generate(rev_prompts, sp_greedy_768)\n\n    print("Generating IoE baseline...")\n    sys.stdout.flush()\n    ioe_outputs = llm.generate(ioe_prompts, sp_greedy_768)\n\n    for idx in range(n_items):\n        r = a0_prompts[idx][0]\n        a0_out = a0_outputs[idx]\n        a0_text = a0_out.outputs[0].text\n        a0_extracted = extract_final_answer(a0_text)\n\n        v_out = verb_outputs[idx]\n        v_text = v_out.outputs[0].text\n        m_int = re.search(r"\\b(\\d+)\\b", v_text)\n        if m_int:\n            val = int(m_int.group(1))\n            val = max(0, min(100, val))\n            s_verb = val / 100.0\n            verb_parse_fail = False\n        else:\n            s_verb = 1.0\n            verb_parse_fail = True\n\n        sc_raw_texts = [\n            sc_sample_outputs[j][idx].outputs[0].text for j in range(10)\n        ]\n        sc_samples_extracted = [\n            extract_final_answer(t) for t in sc_raw_texts\n        ]\n        if dataset_key == "gsm8k":\n            matches = [\n                score_gsm8k(ans, a0_extracted)[0] for ans in sc_samples_extracted\n            ]\n        else:\n            matches = [\n                exact_match_score(ans, a0_extracted)\n                for ans in sc_samples_extracted\n            ]\n        s_sc10 = sum(matches) / 10.0\n        s_sc5 = sum(matches[:5]) / 5.0\n\n        vf_out = verif_outputs[idx]\n        vf_user_msg, vf_proposed, _ = verif_prompts[idx]\n        p_A = 0.0\n        p_B = 0.0\n        if vf_out.outputs[0].logprobs:\n            top_dict = vf_out.outputs[0].logprobs[0]\n            for tok_id, logprob_obj in top_dict.items():\n                tok_str = logprob_obj.decoded_token.strip().upper()\n                prob = math.exp(logprob_obj.logprob)\n                if tok_str == "A":\n                    p_A += prob\n                elif tok_str == "B":\n                    p_B += prob\n        if (p_A + p_B) > 0:\n            s_verif = p_A / (p_A + p_B)\n            verif_parse_fail = False\n        else:\n            s_verif = 0.5\n            verif_parse_fail = True\n\n        a1_out = rev_outputs[idx]\n        a1_text = a1_out.outputs[0].text\n        a1_extracted = extract_final_answer(a1_text)\n\n        ioe_out = ioe_outputs[idx]\n        ioe_text = ioe_out.outputs[0].text\n        ioe_extracted = extract_final_answer(ioe_text)\n\n        maj_extracted = majority_vote([a0_extracted] + sc_samples_extracted)\n\n        gold = r["gold"]\n        if dataset_key == "gsm8k":\n            y0, lenient_y0, f1_y0 = score_gsm8k(a0_extracted, gold)\n            y1, lenient_y1, f1_y1 = score_gsm8k(a1_extracted, gold)\n            y_ioe, lenient_ioe, f1_ioe = score_gsm8k(ioe_extracted, gold)\n            y_major, lenient_major, f1_major = score_gsm8k(maj_extracted, gold)\n        else:\n            y0, lenient_y0, f1_y0 = score_hotpotqa(a0_extracted, gold)\n            y1, lenient_y1, f1_y1 = score_hotpotqa(a1_extracted, gold)\n            y_ioe, lenient_ioe, f1_ioe = score_hotpotqa(ioe_extracted, gold)\n            y_major, lenient_major, f1_major = score_hotpotqa(\n                maj_extracted, gold\n            )\n\n        sc_prompt_len = len(sc_sample_outputs[0][idx].prompt_token_ids)\n        sc_comp_len = sum(\n            len(sc_sample_outputs[j][idx].outputs[0].token_ids) for j in range(10)\n        )\n\n        cost_dict = {\n            "a0_prompt_tokens": len(a0_out.prompt_token_ids),\n            "a0_completion_tokens": len(a0_out.outputs[0].token_ids),\n            "verb_prompt_tokens": len(v_out.prompt_token_ids),\n            "verb_completion_tokens": len(v_out.outputs[0].token_ids),\n            "sc_prompt_tokens": 10 * sc_prompt_len,\n            "sc_completion_tokens": sc_comp_len,\n            "verif_prompt_tokens": len(vf_out.prompt_token_ids),\n            "verif_completion_tokens": len(vf_out.outputs[0].token_ids),\n            "rev_prompt_tokens": len(a1_out.prompt_token_ids),\n            "rev_completion_tokens": len(a1_out.outputs[0].token_ids),\n            "ioe_prompt_tokens": len(ioe_out.prompt_token_ids),\n            "ioe_completion_tokens": len(ioe_out.outputs[0].token_ids),\n        }\n\n        rec_res = {\n            "id": r["id"],\n            "model": model_key,\n            "dataset": dataset_key,\n            "question": r["question"],\n            "context": r["context"],\n            "gold": gold,\n            "a0_text": a0_text,\n            "a0_extracted": a0_extracted,\n            "s_verb": s_verb,\n            "verb_text": v_text,\n            "verb_parse_fail": verb_parse_fail,\n            "s_sc10": s_sc10,\n            "s_sc5": s_sc5,\n            "sc_raw_texts": sc_raw_texts,\n            "sc_extracted_answers": sc_samples_extracted,\n            "s_verif": s_verif,\n            "verif_prompt": vf_user_msg,\n            "verif_proposed_answer": vf_proposed,\n            "verif_parse_fail": verif_parse_fail,\n            "a1_text": a1_text,\n            "a1_extracted": a1_extracted,\n            "ioe_text": ioe_text,\n            "ioe_extracted": ioe_extracted,\n            "majority_extracted": maj_extracted,\n            "y0": y0,\n            "y1": y1,\n            "y_ioe": y_ioe,\n            "y_major": y_major,\n            "lenient_y0": lenient_y0,\n            "lenient_y1": lenient_y1,\n            "f1_y0": f1_y0,\n            "f1_y1": f1_y1,\n            "cost": cost_dict,\n            "duration_seconds": (time.time() - start_total_time) / n_items,\n        }\n        results.append(rec_res)\n\n    return results\n\n\ndef resample_sc_batch(\n    records_to_resample, model_key, dataset_key, llm, tokenizer, prompts\n):\n    """Resample ONLY the 10 SC samples with seeds 42..51, recomputing s_sc5, s_sc10, and majority vote."""\n    results = []\n    start_total_time = time.time()\n    n_items = len(records_to_resample)\n    print(\n        f"Starting SC resample batch for {n_items} records ({model_key}/{dataset_key})..."\n    )\n    sys.stdout.flush()\n\n    sc_prompts = []\n    for r in records_to_resample:\n        if dataset_key == "gsm8k":\n            user_msg = prompts["a0_gsm8k"].format(question=r["question"])\n        else:\n            user_msg = prompts["a0_hotpotqa"].format(\n                context=r.get("context", ""), question=r["question"]\n            )\n        conv = [{"role": "user", "content": user_msg}]\n        text_prompt = tokenizer.apply_chat_template(\n            conv, tokenize=False, add_generation_prompt=True\n        )\n        sc_prompts.append(text_prompt)\n\n    print(\n        f"Generating S_sc (10 separate samples with seeds 42..51 for {n_items} records)..."\n    )\n    sys.stdout.flush()\n    sc_sample_outputs = []  # shape: (10, n_items)\n    for j in range(10):\n        sp_j = SamplingParams(\n            n=1,\n            temperature=0.7,\n            top_p=0.95,\n            seed=42 + j,\n            max_tokens=768,\n        )\n        out_j = llm.generate(sc_prompts, sp_j)\n        sc_sample_outputs.append(out_j)\n\n    for idx, orig_r in enumerate(records_to_resample):\n        # Merge by id: preserve all other fields untouched\n        new_r = dict(orig_r)\n        a0_extracted = new_r["a0_extracted"]\n        gold = new_r["gold"]\n\n        sc_raw_texts = [\n            sc_sample_outputs[j][idx].outputs[0].text for j in range(10)\n        ]\n        sc_samples_extracted = [\n            extract_final_answer(t) for t in sc_raw_texts\n        ]\n        if dataset_key == "gsm8k":\n            matches = [\n                score_gsm8k(ans, a0_extracted)[0] for ans in sc_samples_extracted\n            ]\n        else:\n            matches = [\n                exact_match_score(ans, a0_extracted)\n                for ans in sc_samples_extracted\n            ]\n        s_sc10 = sum(matches) / 10.0\n        s_sc5 = sum(matches[:5]) / 5.0\n\n        maj_extracted = majority_vote([a0_extracted] + sc_samples_extracted)\n        if dataset_key == "gsm8k":\n            y_major, lenient_major, f1_major = score_gsm8k(maj_extracted, gold)\n        else:\n            y_major, lenient_major, f1_major = score_hotpotqa(\n                maj_extracted, gold\n            )\n\n        sc_prompt_len = len(sc_sample_outputs[0][idx].prompt_token_ids)\n        sc_comp_len = sum(\n            len(sc_sample_outputs[j][idx].outputs[0].token_ids)\n            for j in range(10)\n        )\n\n        cost_dict = dict(new_r.get("cost", {}))\n        cost_dict["sc_prompt_tokens"] = 10 * sc_prompt_len\n        cost_dict["sc_completion_tokens"] = sc_comp_len\n\n        new_r["s_sc10"] = s_sc10\n        new_r["s_sc5"] = s_sc5\n        new_r["sc_raw_texts"] = sc_raw_texts\n        new_r["sc_extracted_answers"] = sc_samples_extracted\n        new_r["majority_extracted"] = maj_extracted\n        new_r["y_major"] = y_major\n        new_r["cost"] = cost_dict\n        new_r["duration_seconds"] = (time.time() - start_total_time) / n_items\n\n        results.append(new_r)\n\n    return results\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Run CGSC pipeline for a model/dataset setting"\n    )\n    parser.add_argument(\n        "--model", type=str, required=True, choices=["qwen", "llama"]\n    )\n    parser.add_argument(\n        "--dataset", type=str, required=True, choices=["gsm8k", "hotpotqa"]\n    )\n    parser.add_argument("--dry_count", type=int, default=20)\n    parser.add_argument("--full_count", type=int, default=500)\n    parser.add_argument("--output_dir", type=str, default="/kaggle/working")\n    parser.add_argument(\n        "--resample_sc_only",\n        action="store_true",\n        default=False,\n        help="Resample only the 10 SC samples and recompute s_sc5, s_sc10, majority vote.",\n    )\n    parser.add_argument(\n        "--raw_dir",\n        type=str,\n        default="results/raw",\n        help="Directory containing original raw JSONL files for resampling.",\n    )\n    args = parser.parse_args()\n\n    os.makedirs(args.output_dir, exist_ok=True)\n    setting_tag = f"{args.model}_{args.dataset}"\n    mode_str = "RESAMPLE SC ONLY" if args.resample_sc_only else "FULL PIPELINE"\n    print(f"\\n{\'=\'*70}\\nRUNNING SETTING: {setting_tag} ({mode_str})\\n{\'=\'*70}")\n    sys.stdout.flush()\n\n    # Load prompts\n    prompts_dir = "cgsc/prompts"\n    prompts = load_all_prompts(prompts_dir)\n\n    # Load splits\n    splits_path = "cgsc/data/splits.json"\n    with open(splits_path, "r", encoding="utf-8") as f:\n        splits = json.load(f)\n\n    # Load Model\n    llm, tokenizer = load_model(args.model)\n\n    if args.resample_sc_only:\n        # Load existing JSONL\n        raw_file = os.path.join(args.raw_dir, f"{setting_tag}.jsonl")\n        print(f"Loading existing raw records from {raw_file} for resampling...")\n        sys.stdout.flush()\n        if not os.path.exists(raw_file):\n            raise FileNotFoundError(f"Cannot resample: raw file {raw_file} does not exist!")\n\n        existing_records = []\n        with open(raw_file, "r", encoding="utf-8") as f:\n            for line in f:\n                if line.strip():\n                    existing_records.append(json.loads(line))\n        print(f"Loaded {len(existing_records)} existing records from {raw_file}.")\n\n        # For hotpotqa, ensure context is present (in case compressed raw stripped it)\n        if args.dataset == "hotpotqa":\n            dataset_records = None\n            for r in existing_records:\n                if not r.get("context"):\n                    if dataset_records is None:\n                        dataset_records = get_dataset_records(args.dataset)\n                    if r["id"] in dataset_records:\n                        r["context"] = dataset_records[r["id"]]["context"]\n                if not r.get("verif_prompt"):\n                    r["verif_prompt"] = prompts["verif_hotpotqa"].format(\n                        question=r["question"],\n                        context=r.get("context", ""),\n                        extracted_A0_answer=r.get("a0_extracted", ""),\n                    )\n\n        # 1. DRY RUN\n        dry_records = existing_records[: args.dry_count]\n        print(\n            f"\\n--- Running RESAMPLE DRY RUN ({len(dry_records)} questions) for {setting_tag} ---"\n        )\n        sys.stdout.flush()\n        dry_results = resample_sc_batch(\n            dry_records, args.model, args.dataset, llm, tokenizer, prompts\n        )\n\n        # 2. SANITY CHECKS\n        print(f"\\n--- Running Sanity Checks for {setting_tag} ---")\n        sys.stdout.flush()\n        passed, report = run_sanity_checks(dry_results, args.model, args.dataset)\n        print(report)\n        sys.stdout.flush()\n\n        # Save sanity report\n        report_file = os.path.join(\n            args.output_dir, f"sanity_report_{setting_tag}.txt"\n        )\n        with open(report_file, "w", encoding="utf-8") as f:\n            f.write(report)\n        print(f"Sanity report written to {report_file}")\n        sys.stdout.flush()\n\n        if not passed:\n            err_msg = f"Sanity checks FAILED for {setting_tag}!"\n            print(f"FATAL: {err_msg}")\n            sys.stdout.flush()\n            sys.exit(1)\n\n        # 3. FULL RUN\n        print(f"\\n--- Sanity checks PASSED! Starting FULL RESAMPLE for {setting_tag} ---")\n        sys.stdout.flush()\n        remaining_records = existing_records[args.dry_count : args.full_count]\n        if remaining_records:\n            remaining_results = resample_sc_batch(\n                remaining_records,\n                args.model,\n                args.dataset,\n                llm,\n                tokenizer,\n                prompts,\n            )\n            final_results = dry_results + remaining_results\n        else:\n            final_results = dry_results\n\n    else:\n        # FULL PIPELINE RUN\n        # Load dataset records\n        print(f"Loading dataset records for {args.dataset}...")\n        sys.stdout.flush()\n        dataset_records = get_dataset_records(args.dataset)\n        chosen_ids = splits[args.dataset]["dev"] + splits[args.dataset]["test"]\n        ordered_records = [\n            dataset_records[i] for i in chosen_ids if i in dataset_records\n        ]\n        print(f"Total available ordered records: {len(ordered_records)}")\n        sys.stdout.flush()\n\n        # 1. DRY RUN\n        dry_records = ordered_records[: args.dry_count]\n        print(\n            f"\\n--- Running DRY RUN ({len(dry_records)} questions) for {setting_tag} ---"\n        )\n        sys.stdout.flush()\n        dry_results = run_batch_pipeline(\n            dry_records, args.model, args.dataset, llm, tokenizer, prompts\n        )\n\n        # 2. SANITY CHECKS\n        print(f"\\n--- Running Sanity Checks for {setting_tag} ---")\n        sys.stdout.flush()\n        passed, report = run_sanity_checks(dry_results, args.model, args.dataset)\n        print(report)\n        sys.stdout.flush()\n\n        # Save sanity report\n        report_file = os.path.join(\n            args.output_dir, f"sanity_report_{setting_tag}.txt"\n        )\n        with open(report_file, "w", encoding="utf-8") as f:\n            f.write(report)\n        print(f"Sanity report written to {report_file}")\n        sys.stdout.flush()\n\n        if not passed:\n            err_msg = f"Sanity checks FAILED for {setting_tag}!"\n            print(f"FATAL: {err_msg}")\n            sys.stdout.flush()\n            sys.exit(1)\n\n        # 3. FULL RUN\n        print(f"\\n--- Sanity checks PASSED! Starting FULL RUN for {setting_tag} ---")\n        sys.stdout.flush()\n        remaining_records = ordered_records[args.dry_count : args.full_count]\n        if remaining_records:\n            remaining_results = run_batch_pipeline(\n                remaining_records,\n                args.model,\n                args.dataset,\n                llm,\n                tokenizer,\n                prompts,\n            )\n            final_results = dry_results + remaining_results\n        else:\n            final_results = dry_results\n\n    # 4. Save JSONL\n    jsonl_file = os.path.join(args.output_dir, f"{setting_tag}.jsonl")\n    with open(jsonl_file, "w", encoding="utf-8") as f:\n        for r in final_results:\n            f.write(json.dumps(r, ensure_ascii=False) + "\\n")\n    print(f"Successfully saved {len(final_results)} records to {jsonl_file}")\n    sys.stdout.flush()\n\n    # 5. Create ZIP Archive immediately\n    zip_file = os.path.join(args.output_dir, f"{setting_tag}_bundle.zip")\n    with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as zf:\n        zf.write(jsonl_file, arcname=f"{setting_tag}.jsonl")\n        zf.write(report_file, arcname=f"sanity_report_{setting_tag}.txt")\n    print(f"Created bundle archive at {zip_file}")\n    sys.stdout.flush()\n\n    print(f"\\nSUCCESS: Pipeline finished cleanly for {setting_tag}!")\n    sys.stdout.flush()\n\n\nif __name__ == "__main__":\n    main()\n')

prompt_data = {'a0_gsm8k.txt': 'Solve the following problem step by step. At the end, write your answer on a new line as:\nFinal answer: <answer>\n\nProblem: {question}\n', 'a0_hotpotqa.txt': 'Answer the question using the paragraphs below. Think step by step. At the end, write your answer on a new line as:\nFinal answer: <answer>\nThe answer should be a short span (a name, date, number, or yes/no).\n\nParagraphs:\n{context}\n\nQuestion: {question}\n', 'verb.txt': 'How confident are you that your final answer above is correct? Reply with a single integer from 0 to 100 and nothing else.\n', 'verif_gsm8k.txt': 'Question: {question}\nProposed answer: {extracted_A0_answer}\nIs the proposed answer correct?\n(A) True\n(B) False\nAnswer with the letter only.\n', 'verif_hotpotqa.txt': 'Question: {question}\nParagraphs:\n{context}\nProposed answer: {extracted_A0_answer}\nIs the proposed answer correct?\n(A) True\n(B) False\nAnswer with the letter only.\n', 'revision.txt': 'Review your previous answer carefully. Check each step for errors. If you find a specific error, correct it. If you find no error, keep your original answer. End with a new line: Final answer: <answer>\n', 'ioe.txt': '# Verify wording against Li et al. 2024, IoE prompt, before submission.\nReview your previous answer. If you are very confident about your answer, maintain your answer. Otherwise, update your answer. End with a new line: Final answer: <answer>\n'}
for fname, content in prompt_data.items():
    with open(os.path.join("cgsc/prompts", fname), "w", encoding="utf-8") as f:
        f.write(content)

# Mount original raw Qwen files from attached Kaggle dataset fakharalam1/cgsc-raw
raw_src_dir = "/kaggle/input/cgsc-raw"
for f in ["qwen_gsm8k.jsonl", "qwen_hotpotqa.jsonl"]:
    src_f = os.path.join(raw_src_dir, f)
    dst_f = os.path.join("results/raw", f)
    if os.path.exists(src_f):
        print(f"Copying {f} from {raw_src_dir} to results/raw/...")
        shutil.copy(src_f, dst_f)
    elif os.path.exists(f):
        shutil.copy(f, dst_f)
    else:
        print(f"Warning: {src_f} not found; checking working directory...")

print(f"results/raw contents: {os.listdir('results/raw')}")
print("cgsc package files & datasets successfully initialized on disk.")


In [ ]:
import os, sys, subprocess

print("=" * 70)
print("4. EXECUTING CGSC PIPELINE VIA ISOLATED SUBPROCESS")
print("=" * 70)

# Configure environment for subprocess
sub_env = os.environ.copy()
sub_env["HF_HOME"] = "/tmp/hf"
sub_env["VLLM_USE_V1"] = "0"

# Securely retrieve HF_TOKEN from kaggle_secrets without printing
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_tok = user_secrets.get_secret("HF_TOKEN")
    if hf_tok:
        sub_env["HF_TOKEN"] = hf_tok
        sub_env["HUGGING_FACE_HUB_TOKEN"] = hf_tok
        print("HF_TOKEN successfully loaded from Kaggle secrets.")
    else:
        print("HF_TOKEN secret not found or empty.")
except Exception as e:
    print(f"Notice: UserSecretsClient returned: {e}")

errors_file = os.path.join(OUTPUT_DIR, "errors.txt")

for setting_str in SETTINGS:
    if ":resample" in setting_str:
        base = setting_str.replace(":resample", "")
        model_key, dataset_key = base.split("/")
        is_resample = True
    else:
        model_key, dataset_key = setting_str.split("/")
        is_resample = False

    setting_tag = f"{model_key}_{dataset_key}"
    mode_desc = "RESAMPLE SC ONLY" if is_resample else "FULL PIPELINE"
    print(f"\n{'#' * 70}")
    print(f"# STARTING SETTING: {setting_tag} ({mode_desc})")
    print(f"{'#' * 70}\n")
    sys.stdout.flush()

    cmd = [
        "/tmp/venv/bin/python", "-m", "cgsc.run",
        "--model", model_key,
        "--dataset", dataset_key,
        "--dry_count", str(DRY_RUN_COUNT),
        "--full_count", str(FULL_RUN_COUNT),
        "--output_dir", OUTPUT_DIR,
        "--raw_dir", "results/raw"
    ]
    if is_resample:
        cmd.append("--resample_sc_only")

    res = subprocess.run(cmd, env=sub_env)
    if res.returncode != 0:
        err_msg = f"Setting {setting_tag} failed with exit code {res.returncode}"
        print(f"\nERROR: {err_msg}")
        with open(errors_file, "a", encoding="utf-8") as ef:
            ef.write(f"[{setting_tag}] {err_msg}\n")
        print(f"Skipping {setting_tag} and continuing to next setting.")
    else:
        print(f"\nSUCCESS: Setting {setting_tag} completed successfully!")

    sys.stdout.flush()

print("\n" + "=" * 70)
print("ALL REQUESTED SETTINGS PROCESSED!")
print("=" * 70)
